# Notebook 17 — Appendix: Math Background + Big-Step Semantics

**Covers Appendix A (math background) + Appendix B (big-step operational semantics).**

Two parts:

1. **Appendix A** is mostly review of definitions from COMP11120 (functions, partial functions, relations, countability). We summarise concisely — these underpin chapters 5 and 6.
2. **Appendix B** introduces an *alternative* operational semantics for While: **big-step** (also called *natural semantics*). Same language, different proof style — focuses on the *final state* directly, rather than each step. Useful when you want to reason about what a program *computes* without tracking *how* it does so.

## What you'll be able to do by the end

- Recall the math vocabulary: injection, surjection, bijection, partial function, relation, countability.
- State the 5 big-step rules and apply them to derive `⟨S, σ⟩ ⇓ σ'`.
- Compare big-step and small-step proofs on the same program.
- Solve exercises 58–67.

In [1]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError('could not find while_lang.py')

from while_lang import (
    parse, run, trace, count_steps,
    big_step, big_step_agrees_small_step,
    StepBudgetExceeded,
)
print('imports OK')

imports OK


## §1. Appendix A — math background recap

Concise review of the math vocabulary the chapters use. Most of this is from COMP11120.

### A.1 Numbers — three facts

**Fact 1 (integer division).** For `x, y ∈ ℤ` with `x ≠ 0`, there exist unique `k, l ∈ ℤ` with `0 ≤ l < |x|` and `y = kx + l`. Then `y div x = k` and `y mod x = l`.

**Fact 2 (Unique Prime Factorization, aka the Fundamental Theorem of Arithmetic).** Every `n ∈ ℕ \ {0}` has a unique factorization as a product of primes:

$$n = p_1^{i_1} \cdot p_2^{i_2} \cdots p_k^{i_k}$$

Used in chapter 6 to show that φ is injective: from `2^m (2n+1) = 2^{m'} (2n'+1)` we conclude `m = m'` and `n = n'` because the factorization into a power of 2 and an odd number is unique.

**Fact 3 (real-number ordering).** Standard properties: ≤ is total, addition and positive multiplication preserve ≤, negation reverses, etc. Used throughout chapter 3's complexity proofs.

### A.2 Functions

A function `f : S → T` has a source `S`, a target `T`, and an assignment of one output in `T` to each input in `S`.

| Property | Definition | In words |
|:--|:--|:--|
| **Injective** ("one-to-one") | `f(s) = f(s') ⟹ s = s'` | Different inputs always give different outputs. |
| **Surjective** ("onto") | `∀t ∈ T, ∃s ∈ S, f(s) = t` | Every output value is reached by some input. |
| **Bijective** | both injective and surjective | Pairs up `S` and `T` perfectly. |
| **Inverse** of `f` | `g : T → S` with `g ∘ f = id_S` and `f ∘ g = id_T` | Undoes `f`. |

**Theorem 32:** `f` has an inverse iff `f` is bijective.

Used everywhere — especially the encoding bijections in chapter 6.

### A.3 Partial functions

A **partial function** `f : S ⇀ T` (half-arrow) is an assignment where each input has *at most one* output. Some inputs may have *no* output — those inputs are *not in the domain of f*.

$$\text{dom}(f) = \{s \in S \mid f(s) \text{ is defined}\}$$

Composition of partial functions: `(g ∘ f)(s) = g(f(s))` *if* both are defined; otherwise undefined.

Used in chapter 5 — programs compute *partial* functions because they may not terminate.

### A.4 Binary relations

A **binary relation** `R` on a set `S` is a subset `R ⊆ S × S`.

Four properties:

| Property | Definition |
|:--|:--|
| **Reflexive** | `∀s ∈ S, (s, s) ∈ R` |
| **Symmetric** | `∀s, s' ∈ S, (s, s') ∈ R ⟹ (s', s) ∈ R` |
| **Antisymmetric** | `(s, s') ∈ R ∧ (s', s) ∈ R ⟹ s = s'` |
| **Transitive** | `(s, s') ∈ R ∧ (s', s'') ∈ R ⟹ (s, s'') ∈ R` |

**Closures:**
- *Reflexive closure*: `R ∪ {(s, s) : s ∈ S}`
- *Transitive closure*: smallest transitive relation containing `R`. Recursively: `R^0 = id_S, R^{n+1} = R^n ; R`. Then `⋃ R^n` for n ≥ 1.
- *Reflexive-transitive closure*: `⋃ R^n` for n ≥ 0. This is the `⇒*` we use everywhere for the small-step relation.

### A.5 Countable and uncountable

**Definition 38.** `|S| ≤ |T|` iff there's an injection from S to T.

**Definition 41.** S is **countable** iff there's an injection S → ℕ. **Uncountable** otherwise. **Countably infinite** iff countable AND infinite.

**Examples:**

| Countably infinite | Uncountable |
|:--|:--|
| ℕ, ℤ, ℚ | ℝ, ℂ |
| Finite subsets of ℕ | All subsets P(ℕ) |
| Programs in any reasonable language | Fun(ℕ, ℕ) |
| Strings over a finite alphabet | Real interval `[0, 1]` |

Used in chapter 5 to argue most functions are uncomputable: the set of programs is countable, but the function space is uncountable, so there's a mismatch.

## §2. Appendix B — big-step operational semantics

**Recall:** the *small-step* semantics (chapter 2) tracks every transition `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` until reaching `⟨skip, σ'⟩`. **Big-step** is different: it directly relates a starting configuration to its final state.

$$\langle S, \sigma\rangle \Downarrow \sigma'$$

**Read:** "executing S in state σ terminates in state σ'."

Big-step is good for proving facts about *what a program computes* (the final state). It's worse than small-step for tracking individual steps (that's what small-step is for).

### The 5 rules

**Rule 1 — assignment**

$$\langle x := a, \sigma\rangle \Downarrow \sigma[x \mapsto \mathcal{A}(a, \sigma)]$$

Same as small-step's `:=` rule, just collapsed.

**Rule 2 — skip**

$$\langle \textbf{skip}, \sigma\rangle \Downarrow \sigma$$

Skip preserves the state.

**Rule 3 — composition**

$$\dfrac{\langle S, \sigma\rangle \Downarrow \sigma' \qquad \langle S', \sigma'\rangle \Downarrow \sigma''}{\langle S; S', \sigma\rangle \Downarrow \sigma''}$$

Run the first, then the second from the resulting state. **The crucial difference from small-step's `;` rule:** big-step requires the first half to *terminate* (reach σ') before the second half starts. There's no "middle of S" state.

**Rule 4 — conditional (two cases)**

$$\dfrac{\langle S, \sigma\rangle \Downarrow \sigma'}{\langle \textbf{if } b \textbf{ then } S \textbf{ else }(S'), \sigma\rangle \Downarrow \sigma'} \quad \text{if } \mathcal{B}(b, \sigma) = \mathbf{tt}$$

$$\dfrac{\langle S', \sigma\rangle \Downarrow \sigma''}{\langle \textbf{if } b \textbf{ then } S \textbf{ else }(S'), \sigma\rangle \Downarrow \sigma''} \quad \text{if } \mathcal{B}(b, \sigma) = \mathbf{ff}$$

**Rule 5 — while (two cases)**

$$\dfrac{\langle S, \sigma\rangle \Downarrow \sigma' \qquad \langle \textbf{while } b \textbf{ do } S, \sigma'\rangle \Downarrow \sigma''}{\langle \textbf{while } b \textbf{ do } S, \sigma\rangle \Downarrow \sigma''} \quad \text{if } \mathcal{B}(b, \sigma) = \mathbf{tt}$$

$$\langle \textbf{while } b \textbf{ do } S, \sigma\rangle \Downarrow \sigma \quad \text{if } \mathcal{B}(b, \sigma) = \mathbf{ff}$$

**The recursion in the while-tt case is the trickiest part.** To prove `⟨while ..., σ⟩ ⇓ σ''`, we need to *already know* `⟨while ..., σ'⟩ ⇓ σ''` for some intermediate σ'. That's what makes big-step proofs feel "backwards" — you reason from the final state back through iterations.

### Big-step in our interpreter

We've added `big_step(prog, state)` to `while_lang.py`. It directly returns the final state.

In [2]:
# Demonstrate big_step.
div = '''
    r := m;
    d := 0;
    while n <= r do (
        d := d + 1;
        r := r - n
    )
'''

print('division program (big-step) on m=10, n=3:')
print(' ', big_step(div, {'m': 10, 'n': 3}))
print('  small-step run() agrees:', big_step_agrees_small_step(div, {'m': 10, 'n': 3}))

print()
for prog, st, label in [
    ('skip', {}, 'skip'),
    ('x := 5', {}, 'single assignment'),
    ('x := 5; y := x + 1', {}, 'sequential'),
    ('if 0 <= x then y := 1 else (y := 2)', {'x': -3}, 'if (false branch)'),
    ('while 1 <= n do (n := n - 1)', {'n': 10}, 'while (counts down)'),
]:
    big = big_step(prog, st)
    small = run(prog, st)
    agree = big == small
    print(f'  {label:30}: big = {big}  small = {small}  agree = {agree}')

division program (big-step) on m=10, n=3:
  {'m': 10, 'n': 3, 'r': 1, 'd': 3}
  small-step run() agrees: True

  skip                          : big = {}  small = {}  agree = True
  single assignment             : big = {'x': 5}  small = {'x': 5}  agree = True
  sequential                    : big = {'x': 5, 'y': 6}  small = {'x': 5, 'y': 6}  agree = True
  if (false branch)             : big = {'x': -3, 'y': 2}  small = {'x': -3, 'y': 2}  agree = True
  while (counts down)           : big = {}  small = {}  agree = True


## §3. Big-step vs small-step — when to use which

| Aspect | Small-step (Chapter 2) | Big-step (Appendix B) |
|:--|:--|:--|
| What's tracked | Each individual transition | Just the final state |
| Notation | `⟨S, σ⟩ ⇒ ⟨S', σ'⟩` | `⟨S, σ⟩ ⇓ σ'` |
| Good for | Counting steps; analysing intermediate states; complexity | Proving termination + final-state properties |
| Reasoning style | Forward (start → step → step → done) | Recursive / backwards (assume the recursive call terminates, derive the conclusion) |
| While loops | Unfold via `while-tt` rule into `body; loop` | Recurse on `loop` from the post-body state |
| Diverging programs | Trace is infinite; reflexive-transitive closure doesn't reach skip | No derivation tree exists; system silent on non-termination |

**Key result connecting the two (Exercise 67):**

$$\langle S, \sigma\rangle \Downarrow \sigma' \quad \iff \quad \langle S, \sigma\rangle \Rightarrow^* \langle \textbf{skip}, \sigma'\rangle$$

So both semantics agree on what "the program terminates with this final state" means. They just present the proof differently.

### When you'd reach for which

- **"How many steps?"** → small-step (literally counts).
- **"Does this loop preserve invariant I?"** → big-step often easier (one assumption per recursion).
- **"Does this fail to terminate?"** → small-step (you can keep applying the rule and watch for loops).
- **"What's the final state?"** → either, but big-step gets you there in fewer steps.

## §4. Worked example — big-step derivation of the division program

Following Appendix B.2: derive `⟨r := m; d := 0; while n ≤ r do B, σ⟩ ⇓ σ'` for σ = `{m ↦ 10, n ↦ 3}`.

We use **B = `d := d + 1; r := r − n`** for the body, **L = `while n ≤ r do (B)`**.

**Linearised derivation (numbered lines, each citing the rule and previous lines):**

```
1. ⟨r := m, σ⟩ ⇓ σ[r ↦ 10]                                    by := rule.
2. ⟨d := 0, σ[r ↦ 10]⟩ ⇓ σ[r ↦ 10, d ↦ 0]                      by := rule.
3. ⟨L, σ[r ↦ 1, d ↦ 3]⟩ ⇓ σ[r ↦ 1, d ↦ 3]                      by while-ff (since 3 ≤ 1 is false).
4. ⟨B, σ[r ↦ 4, d ↦ 2]⟩ ⇓ σ[r ↦ 1, d ↦ 3]                      by := × 2 + ; (one body iteration).
5. ⟨L, σ[r ↦ 4, d ↦ 2]⟩ ⇓ σ[r ↦ 1, d ↦ 3]                      by while-tt, from 4 and 3.
6. ⟨B, σ[r ↦ 7, d ↦ 1]⟩ ⇓ σ[r ↦ 4, d ↦ 2]                      by := × 2 + ;
7. ⟨L, σ[r ↦ 7, d ↦ 1]⟩ ⇓ σ[r ↦ 1, d ↦ 3]                      by while-tt, from 6 and 5.
8. ⟨B, σ[r ↦ 10, d ↦ 0]⟩ ⇓ σ[r ↦ 7, d ↦ 1]                     by := × 2 + ;
9. ⟨L, σ[r ↦ 10, d ↦ 0]⟩ ⇓ σ[r ↦ 1, d ↦ 3]                     by while-tt, from 8 and 7.
10. ⟨r := m; d := 0; L, σ⟩ ⇓ σ[r ↦ 1, d ↦ 3]                   by ; rule applied to 1, 2, 9.
```

**Notice the working order.** We *start* with the smallest piece — the loop in its terminating state (line 3, since `3 ≤ 1` is false, while-ff applies). Then we work *outward* through previous iterations, building up to the full loop derivation in line 9. Finally we glue with the prefix assignments via `;` rule.

**The big-step style is recursive-feels-backwards.** This is why finding loop invariants and variants makes it easier to *predict* the post-state — you derive the recursion only after knowing where it lands.

## §5. Exercises 58–67

## Exercise 58 — programs without loops always terminate

> Use the big-step semantics to argue that a program built from assignments, skip, and if-statements (composed with `;`) terminates for every start state.

**Argument (induction on program structure):**

We want to show: for any such program `S` and any state `σ`, there exists `σ'` with `⟨S, σ⟩ ⇓ σ'`.

**Base cases:**

1. `S = skip`: `⟨skip, σ⟩ ⇓ σ` by the skip rule. Take `σ' = σ`. ✓
2. `S = x := a`: `⟨x := a, σ⟩ ⇓ σ[x ↦ A(a, σ)]` by the := rule. ✓

**Inductive cases:**

3. `S = S₁; S₂`: by IH, ∃σ' with `⟨S₁, σ⟩ ⇓ σ'`. By IH again, ∃σ'' with `⟨S₂, σ'⟩ ⇓ σ''`. Apply ; rule to conclude `⟨S₁; S₂, σ⟩ ⇓ σ''`. ✓
4. `S = if b then S₁ else (S₂)`:
   - If `B(b, σ) = tt`: by IH, ∃σ' with `⟨S₁, σ⟩ ⇓ σ'`. Apply if-tt rule.
   - If `B(b, σ) = ff`: by IH, ∃σ' with `⟨S₂, σ⟩ ⇓ σ'`. Apply if-ff rule.

**There's no while case** because we excluded loops by hypothesis. So the induction covers every form. **Every loop-free program terminates.** ∎

**Why this matters:** loops are *the* source of non-termination. Without them, programs are guaranteed to halt. (Adding back loops is what introduces the Halting Problem.)

## Exercise 59 — big-step for the Diophantine searcher (Ex 7) on σ = {m=3, n=4, k=7}

From N3 we know the program (the Ex 7 solution from N2) terminates with `x = 1, y = 1` for these inputs. Let's verify with `big_step`.

In [3]:
diophantine = '''
    x := 0; y := 0;
    while !(m * x + n * y = k) do (
        x := x + 1;
        y := 0;
        while !(m * x + n * y = k) & y <= x & !(y = x) do (
            if m * y + n * x = k then (
                z := x;
                x := y;
                y := z
            ) else (
                y := y + 1
            )
        )
    )
'''

final = big_step(diophantine, {'m': 3, 'n': 4, 'k': 7})
print('Diophantine on (3, 4, 7) — big-step final state:', final)
print('Verifies 3·1 + 4·1 = 7:', 3 * final.get('x', 0) + 4 * final.get('y', 0) == 7)

# Compare with N3's small-step result
from while_lang import run
print('Matches small-step run():', big_step(diophantine, {'m': 3, 'n': 4, 'k': 7}) == run(diophantine, {'m': 3, 'n': 4, 'k': 7}))

Diophantine on (3, 4, 7) — big-step final state: {'m': 3, 'n': 4, 'k': 7, 'x': 1, 'y': 1}
Verifies 3·1 + 4·1 = 7: True
Matches small-step run(): True


## Exercise 60 — big-step for gcd on (m=12, n=30)

Same approach. gcd from Example 3, computed via big-step.

In [4]:
gcd_prog = '''
    x := m; y := n;
    while !(x = y) do (
        if !(x <= y) then
            x := x - y
        else (
            y := y - x
        )
    )
'''

final = big_step(gcd_prog, {'m': 12, 'n': 30})
print('gcd(12, 30) — big-step final state:', final)
print('Expected x = y = 6 (the gcd of 12 and 30):',
      final.get('x', 0) == 6 and final.get('y', 0) == 6)

gcd(12, 30) — big-step final state: {'m': 12, 'n': 30, 'x': 6, 'y': 6}
Expected x = y = 6 (the gcd of 12 and 30): True


## Exercise 61 — two programs computing triangular numbers

**Program left** (returns sum 1+2+...+i in `a`):
```
y := 1; a := 0;
while y ≤ i do (a := a + y; y := y + 1)
```

**Program right** (returns 1 + i(i+1)/2 in `y`):
```
x := i × (i+1); y := 0;
while 0 ≤ x do (x := x − 2; y := y + 1)
```

### (a) Small-step both programs from σ = {i = 5}

Both compute related quantities. We've already covered this in N6 (Exercise 19) — both terminate, but with different complexity (left is O(n), right is O(n²)).

### (b) Big-step for program left

**Sketch.** The loop runs i times. Working backwards: when y reaches i+1, the loop terminates with `a = 1 + 2 + ... + i = i(i+1)/2`.

Big-step derivation skeleton (for i = 5):
1. `⟨body, σ[a ↦ 1+...+i, y ↦ i+1]⟩ ⇓ ...` — but we need a base case where the loop *exits*.
2. By while-ff, `⟨L, σ[a ↦ 15, y ↦ 6]⟩ ⇓ σ[a ↦ 15, y ↦ 6]`.
3. Then build up backwards through 5 iterations.

Easier in our interpreter:

In [5]:
left_prog = '''
    y := 1; a := 0;
    while y <= i do (
        a := a + y;
        y := y + 1
    )
'''
right_prog = '''
    x := i * (i + 1); y := 0;
    while 0 <= x do (
        x := x - 2;
        y := y + 1
    )
'''

for i in [3, 5, 10]:
    left = big_step(left_prog, {'i': i})
    right = big_step(right_prog, {'i': i})
    expected_T = i * (i + 1) // 2
    print(f'i = {i}: left.a = {left.get("a", 0)} (expected T_i = {expected_T}); right.y = {right.get("y", 0)} (expected T_i + 1 = {expected_T + 1})')

i = 3: left.a = 6 (expected T_i = 6); right.y = 7 (expected T_i + 1 = 7)
i = 5: left.a = 15 (expected T_i = 15); right.y = 16 (expected T_i + 1 = 16)
i = 10: left.a = 55 (expected T_i = 55); right.y = 56 (expected T_i + 1 = 56)


### (c) Formal equivalence statement

Both programs *eventually* express the same `i(i+1)/2`, just shifted by one (right-program ends with y = T_i + 1, not T_i). For exact equivalence with respect to the *output variable*, we'd state:

$$\forall \sigma. \sigma(i) \ge 0 \implies \big(\langle \text{left}, \sigma\rangle \Downarrow \sigma' \implies \sigma'(a) = \sigma(i)(\sigma(i)+1)/2\big)$$

Similarly for `right`, with `σ'(y) = σ(i)(σ(i)+1)/2 + 1`. So they differ by 1 — not literally the same, but related.

## Exercise 62 — when does `⟨while b do (S), σ⟩ ⇓ τ` hold?

**Two cases:**

1. **Loop never enters body.** `B(b, σ) = ff`. Then by while-ff, `⟨while b do S, σ⟩ ⇓ σ`. So `τ = σ`.

2. **Loop enters at least once.** `B(b, σ) = tt`. By while-tt, we need:
   - `⟨S, σ⟩ ⇓ σ'` for some intermediate σ' (one body iteration).
   - `⟨while b do S, σ'⟩ ⇓ τ` (the loop terminating from σ').

**Combined:** `⟨while b do S, σ⟩ ⇓ τ` iff there's a finite sequence `σ = σ_0, σ_1, σ_2, ..., σ_k = τ` such that:
- For each `i < k`: `B(b, σ_i) = tt` and `⟨S, σ_i⟩ ⇓ σ_{i+1}` (one body iteration each).
- `B(b, σ_k) = ff` (loop exits when reaching σ_k).

This is the iteration version of the recursive while-tt rule. The body must terminate at every step, and the loop condition must eventually become false.

**Conclusion:** `⟨while b do S, σ⟩ ⇓ τ` iff the loop runs through finitely many iterations, each terminating, with the loop condition eventually becoming false.

## Exercise 63 — Exercise 4's no-if encoding behaves like the original (big-step)

Compare to N3's Exercise 15 (which used small-step). Same claim, big-step proof — usually cleaner.

**Claim:** the no-if encoding (using two while loops + a flag) and the original `if b then S else (S')` produce the same final state, *if the original terminates*.

**Proof sketch (big-step):**

Recall the encoding:
```
done := 0;
while b ∧ done = 0 do (S; done := 1);
while ¬(done = 1) ∧ ¬b do (S'; done := 1)
```

**Case 1: `B(b, σ) = tt`.** Original: `⟨if b then S else S', σ⟩ ⇓ σ'` where `⟨S, σ⟩ ⇓ σ'` (assuming S terminates). Encoding:
- After `done := 0`: state is σ[done ↦ 0].
- First while: `b ∧ done = 0 = tt ∧ tt = tt`. Body fires: run S then `done := 1`. So body produces σ'[done ↦ 1].
- After body: condition is `b ∧ done = 0` evaluated in σ'[done ↦ 1] — `done = 0` is false (since done = 1), so loop terminates.
- Second while: condition is `¬(done = 1) ∧ ¬b` in σ'[done ↦ 1] — `¬(done = 1)` is false, loop terminates without entering body.
- Final: σ'[done ↦ 1]. Modulo the ghost variable `done`, this matches the original σ'.

**Case 2: `B(b, σ) = ff`.** Symmetric — the second while runs once, doing `S'; done := 1`, while the first does nothing. Final state matches the original.

**Why big-step is cleaner here:** in small-step, you need to track the iterations of both whiles individually. Big-step lets you reason about "the loop terminates" as a single unit.

Empirical sanity check:

In [6]:
original = 'if x = 0 then (y := 1) else (y := 2)'
no_if = '''
    done := 0;
    while x = 0 & done = 0 do (
        y := 1;
        done := 1
    );
    while !(done = 1) & !(x = 0) do (
        y := 2;
        done := 1
    )
'''

for x in [-3, 0, 7]:
    a = big_step(original, {'x': x})
    b = big_step(no_if, {'x': x})
    b_minus_done = {k: v for k, v in b.items() if k != 'done'}
    print(f'x = {x:>3}: original = {a},  no-if (sans done) = {b_minus_done},  agree = {a == b_minus_done}')

x =  -3: original = {'x': -3, 'y': 2},  no-if (sans done) = {'x': -3, 'y': 2},  agree = True
x =   0: original = {'y': 1},  no-if (sans done) = {'y': 1},  agree = True
x =   7: original = {'x': 7, 'y': 2},  no-if (sans done) = {'x': 7, 'y': 2},  agree = True


## Exercise 64 — big-step rules for `repeat S until b`

### (a) The rules

**Recall:** `repeat S until b` runs `S` *first*, then checks `b`. Loops until `b` is true.

**Two cases (parallel to while):**

**Case 1 — body terminates with `b` becoming true:**

$$\dfrac{\langle S, \sigma\rangle \Downarrow \sigma'}{\langle \textbf{repeat } S \textbf{ until } b, \sigma\rangle \Downarrow \sigma'} \quad \text{if } \mathcal{B}(b, \sigma') = \mathbf{tt}$$

**Case 2 — body terminates but `b` is false; loop continues:**

$$\dfrac{\langle S, \sigma\rangle \Downarrow \sigma' \qquad \langle \textbf{repeat } S \textbf{ until } b, \sigma'\rangle \Downarrow \sigma''}{\langle \textbf{repeat } S \textbf{ until } b, \sigma\rangle \Downarrow \sigma''} \quad \text{if } \mathcal{B}(b, \sigma') = \mathbf{ff}$$

Note the asymmetry compared to while: the *body always runs at least once* (even if b is true initially), so there's no "skip" case.

### (b) Equivalence with the Ex 3 encoding

Recall Ex 3 says `repeat S until b` ≡ `S; while ¬b do S`.

**Claim:** `⟨repeat S until b, σ⟩ ⇓_R σ'` iff `⟨S; while ¬b do S, σ⟩ ⇓ σ'`.

**Proof sketch (induction on the number of body executions).** Both unfoldings produce the same sequence of body executions: run S once, check b (or ¬b after one execution), loop if not done.

- Base case (body runs exactly once): repeat-rule with `B(b, σ') = tt` produces σ'. Encoded version: `S; while ¬b do S` runs S to σ', then checks `¬b` (= ff), so while-ff fires; final state σ'. ✓
- Step case (body runs k+1 times): repeat-rule case 2 reduces to `repeat S until b` from σ' — which by IH equals `S; while ¬b do S` from σ' (k iterations). The encoded version: after running S to σ', we enter the while loop; the loop runs ¬b k more times before exiting. Same final state. ✓

Both produce the same state in the same number of body executions. **Equivalent.** ∎

## Exercise 65 — big-step is functional (deterministic)

**Claim:** for every program S and state σ, there is at most one σ' with `⟨S, σ⟩ ⇓ σ'`.

**Proof (induction on the structure of S):**

**Base cases:**
1. `S = skip`: only the skip rule applies, producing σ. Unique.
2. `S = x := a`: only the := rule applies, producing σ[x ↦ A(a, σ)]. Since A is a function, the result is uniquely determined.

**Inductive cases:**
3. `S = S₁; S₂`: only the ; rule applies. By IH, the σ' from `⟨S₁, σ⟩ ⇓ σ'` is unique. Then by IH again, the σ'' from `⟨S₂, σ'⟩ ⇓ σ''` is unique. So `⟨S₁; S₂, σ⟩ ⇓ σ''` is unique.
4. `S = if b then S₁ else S₂`: only one of if-tt or if-ff applies, depending on `B(b, σ)` (which is unique). The result is then determined by IH on the branch.
5. `S = while b do S'`: only one of while-tt or while-ff applies, depending on `B(b, σ)`. In the false case, σ is the unique result. In the true case, by IH the body produces a unique σ', then by IH (recursive call on the same loop) the rest produces a unique σ''.

All cases give unique results. ∎

**Note:** this is the big-step analog of small-step's determinism (Exercise 18). Both semantics are deterministic — same input gives same output.

## Exercise 66 — `;` is associative (big-step)

**Claim:** `⟨(S; T); U, σ⟩ ⇓ σ'` iff `⟨S; (T; U), σ⟩ ⇓ σ'`.

**Proof.** Both reduce to the same chain of body executions:

**Forward direction:** assume `⟨(S; T); U, σ⟩ ⇓ σ'`. By the ; rule, there's σ₁ with `⟨S; T, σ⟩ ⇓ σ₁` and `⟨U, σ₁⟩ ⇓ σ'`. By ; rule again on the inner one, there's σ₂ with `⟨S, σ⟩ ⇓ σ₂` and `⟨T, σ₂⟩ ⇓ σ₁`. Apply ; in the other direction: `⟨T; U, σ₂⟩ ⇓ σ'` (combining `⟨T, σ₂⟩ ⇓ σ₁` and `⟨U, σ₁⟩ ⇓ σ'`). Then `⟨S; (T; U), σ⟩ ⇓ σ'` (combining `⟨S, σ⟩ ⇓ σ₂` and the previous). ✓

**Reverse direction:** symmetric.

**Big-step makes this obvious.** The ; rule's premise structure is a chain that doesn't care about the parenthesisation. Compare to small-step (Proposition 3) where the proof was longer because we had to track step-by-step transitions through both groupings. ∎

## Exercise 67 — big-step ⟺ small-step on terminating programs

**Claim:** `⟨S, σ⟩ ⇓ σ'` iff `⟨S, σ⟩ ⇒* ⟨skip, σ'⟩`.

**This is the formal connection between the two semantics.** Both agree on what "S terminates with final state σ'" means.

**Forward direction (⇓ ⟹ ⇒*):** structural induction on the big-step derivation:
- Base cases (skip, :=): one small-step rule produces ⟨skip, σ'⟩ directly.
- ;: by IH, `⟨S₁, σ⟩ ⇒* ⟨skip, σ'⟩` and `⟨S₂, σ'⟩ ⇒* ⟨skip, σ''⟩`. By small-step's ; rule and the skip-; rule, `⟨S₁; S₂, σ⟩ ⇒* ⟨skip; S₂, σ'⟩ ⇒ ⟨S₂, σ'⟩ ⇒* ⟨skip, σ''⟩`.
- if/while: similar.

**Reverse direction (⇒* ⟹ ⇓):** induction on the length of the small-step derivation. Each step in the small-step trace can be "absorbed" into building a piece of a big-step derivation tree.

**The proof requires tedious induction, but the principle is simple:** big-step's `;` rule mirrors small-step's iteration of `;`-then-skip-; rules. Both produce the same input/output behaviour. ∎

Empirical sanity check across a range of programs:

In [7]:
from while_lang import big_step_agrees_small_step, big_step_steps, run

# Terminating test cases — both semantics should agree.
test_cases = [
    ('skip', {}),
    ('x := 5; y := x + 1', {}),
    ('if 0 = 0 then x := 1 else (x := 2)', {}),
    ('while 1 <= n do (n := n - 1)', {'n': 5}),
    ('while 1 <= n do (x := x + n; n := n - 1)', {'n': 10}),
]

all_ok = True
for p, s in test_cases:
    if not big_step_agrees_small_step(p, s, max_steps=10_000):
        print(f'  DISAGREEMENT on {p}')
        all_ok = False
print(f'Big-step and small-step agree on {len(test_cases)} normal-depth test cases: {all_ok}')

# Deep-recursion case: gcd(12, 30) takes many while iterations.
# Use big_step_steps which raises Python's recursion limit.
deep_prog = '''
    x := m; y := n;
    while !(x = y) do (
        if !(x <= y) then
            x := x - y
        else (
            y := y - x
        )
    )
'''
big = big_step_steps(deep_prog, {'m': 12, 'n': 30}, max_depth=20_000)
small = run(deep_prog, {'m': 12, 'n': 30}, max_steps=20_000)
print(f'Deep gcd case (big_step_steps): big = {big}, small = {small}, agree = {big == small}')

Big-step and small-step agree on 5 normal-depth test cases: True
Deep gcd case (big_step_steps): big = {'m': 12, 'n': 30, 'x': 6, 'y': 6}, small = {'m': 12, 'n': 30, 'x': 6, 'y': 6}, agree = True


## Summary

**Appendix A** is the math vocabulary: functions (injective/surjective/bijective), partial functions, relations (with closures), countability. Most of this is from COMP11120; chapter 5 and 6 use it heavily.

**Appendix B** introduces big-step semantics — an alternative to chapter 2's small-step:

- Notation: `⟨S, σ⟩ ⇓ σ'` (program S terminates in state σ').
- 5 rules: skip, :=, ; (compose two derivations via shared middle σ'), if (two cases), while (two cases — base when b is false, recursion when true).
- **Key result (Ex 67):** `⟨S, σ⟩ ⇓ σ'` iff `⟨S, σ⟩ ⇒* ⟨skip, σ'⟩`. Both semantics agree on what termination means.

**When to use which:**
- Counting steps, complexity → small-step.
- Reasoning about final-state properties (esp. with proofs) → big-step.
- Both work for proving termination + correctness; the mechanics differ.

**The story now:** with appendices in place, we have:
- Two operational semantics (small-step, big-step) — chapter 2 + appendix B.
- A complexity-counting framework — chapter 3.
- A correctness-proving framework — chapter 4.
- A computability theory — chapter 5.
- The encoding plumbing — chapter 6.
- The math foundations — appendix A.

**The course is complete.**